# Caso 2 - Clasificación: incumplimiento de tarjeta de crédito (UCI Credit Card)
Regresión Logística vs Decision Tree Classifier.

In [1]:
# Funciones auxiliares — Problema 2: Clasificación (UCI Credit Card)
# Son las mismas que usamos en los laboratorios de regresión logística y árboles (caso Titanic).
# Ejecuta esta celda una vez; después sólo llama a las funciones.
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score)
from sklearn.preprocessing import StandardScaler


def plot_frequencies(df, columns, top_n=None):
    """Gráfica de barras con la frecuencia de cada categoría (top_n limita a las más comunes)."""
    for col in columns:
        freq = df[col].value_counts()
        if top_n:
            freq = freq.head(top_n)
        freq_df = freq.rename_axis(col).reset_index(name='Frecuencia')

        title = f'Frecuencias de {col}'
        if top_n and df[col].nunique() > top_n:
            title += f' (top {top_n})'

        fig = px.bar(freq_df, x=col, y='Frecuencia', title=title)
        fig.update_layout(xaxis={'categoryorder': 'total descending'})
        fig.show()


def escalar(X_train, X_test):
    """Estandariza (media 0, desviación 1) ajustando sólo con train. Regresa ambos DataFrames."""
    escalador = StandardScaler().fit(X_train)
    X_train_s = pd.DataFrame(escalador.transform(X_train), columns=X_train.columns, index=X_train.index)
    X_test_s = pd.DataFrame(escalador.transform(X_test), columns=X_test.columns, index=X_test.index)
    return X_train_s, X_test_s


def evaluar_clasificador(nombre, y_true, y_pred):
    """Imprime y regresa exactitud (accuracy), precisión, recall y F1.

    Con dos clases, precisión, recall y F1 se calculan para la clase positiva (1).
    Con más de dos clases son el promedio simple de las clases (average='macro').
    """
    binario = len(np.unique(np.concatenate([np.asarray(y_true), np.asarray(y_pred)]))) <= 2
    promedio = 'binary' if binario else 'macro'
    metricas = {
        'modelo': nombre,
        'exactitud': round(accuracy_score(y_true, y_pred), 4),
        'precisión': round(precision_score(y_true, y_pred, average=promedio, zero_division=0), 4),
        'recall': round(recall_score(y_true, y_pred, average=promedio, zero_division=0), 4),
        'F1': round(f1_score(y_true, y_pred, average=promedio, zero_division=0), 4),
    }
    print(f"{nombre}: exactitud {metricas['exactitud']} | precisión {metricas['precisión']} "
          f"| recall {metricas['recall']} | F1 {metricas['F1']}")
    return metricas


def plot_matriz_confusion(y_true, y_pred, clases=None, etiquetas=None, title='Matriz de confusión'):
    """Aciertos y errores del clasificador: la diagonal son los aciertos.

    clases: los valores de la variable objetivo en el orden en que quieres verlos, p. ej. [0, 1].
            Si no las pasas, se ordenan de menor a mayor.
    etiquetas: nombres a mostrar para esas clases (opcional), p. ej. ['Pagó', 'Incumplió'].
    """
    if clases is None:
        clases = sorted(set(np.asarray(y_true)) | set(np.asarray(y_pred)))
    clases = list(clases)
    etiquetas = list(etiquetas) if etiquetas is not None else clases

    cm = pd.DataFrame(
        confusion_matrix(y_true, y_pred, labels=clases),
        index=[f'Real: {e}' for e in etiquetas],
        columns=[f'Predicho: {e}' for e in etiquetas]
    )

    fig = px.imshow(cm, text_auto=True, color_continuous_scale='Blues', title=title)
    fig.update_layout(width=650, height=450)
    fig.show()
    return cm


In [2]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, recall_score, precision_score, confusion_matrix, ConfusionMatrixDisplay

df = pd.read_csv('https://raw.githubusercontent.com/jorgermzg15/data-files/refs/heads/main/UCI_Credit_Card.csv', index_col='ID')
df = df.rename(columns={'default.payment.next.month': 'IsDefaulter'})
print(df.shape)
print('Clase positiva (IsDefaulter=1):', (df['IsDefaulter'] == 1).sum())

(30000, 24)
Clase positiva (IsDefaulter=1): 6636


In [3]:
df['EDUCATION'] = df['EDUCATION'].replace({0: 4, 5: 4, 6: 4})
print('EDUCATION = 4:', (df['EDUCATION'] == 4).sum())
df['MARRIAGE'] = df['MARRIAGE'].replace({0: 3})
print('MARRIAGE = 3:', (df['MARRIAGE'] == 3).sum())
print('Edad promedio de incumplidores: %.2f' % df.loc[df['IsDefaulter'] == 1, 'AGE'].mean())

EDUCATION = 4: 468
MARRIAGE = 3: 377
Edad promedio de incumplidores: 35.73


In [4]:
pay = ['PAY_0','PAY_2','PAY_3','PAY_4','PAY_5','PAY_6']
n0 = df.shape[1]
df = pd.get_dummies(df, columns=['EDUCATION', 'MARRIAGE'], dtype=int)
print('Dummies EDUCATION + MARRIAGE:', df.shape[1] - (n0 - 2))
n1 = df.shape[1]
df = pd.get_dummies(df, columns=pay, drop_first=True, dtype=int)
print('Dummies PAY_*:', df.shape[1] - (n1 - len(pay)))
if 'ID' in df.columns: df = df.drop(columns='ID')
print('Columnas finales:', df.shape[1])

Dummies EDUCATION + MARRIAGE: 7
Dummies PAY_*: 58
Columnas finales: 81


In [5]:
X = df.drop(columns='IsDefaulter'); y = df['IsDefaulter']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42, stratify=df.IsDefaulter)
print('len(X_train) =', len(X_train))

len(X_train) = 20100


In [6]:
X_train_s, X_test_s = escalar(X_train, X_test)
log = LogisticRegression(C=10, random_state=42, class_weight='balanced', max_iter=2000).fit(X_train_s, y_train)
p_log = log.predict(X_test_s)
m_log = evaluar_clasificador('Regresión Logística', y_test, p_log)

Regresión Logística: exactitud 0.7754 | precisión 0.4932 | recall 0.5658 | F1 0.527


In [7]:
cm = plot_matriz_confusion(y_test, p_log, clases=[0, 1], etiquetas=['Pagó', 'Incumplió'], title='Matriz de confusión - Regresión Logística')
print(cm)

                 Predicho: Pagó  Predicho: Incumplió
Real: Pagó                 6437                 1273
Real: Incumplió             951                 1239


In [8]:
tree = DecisionTreeClassifier(max_depth=20, min_samples_split=0.1, random_state=42, class_weight='balanced').fit(X_train, y_train)
p_tree = tree.predict(X_test)
m_tree = evaluar_clasificador('Decision Tree', y_test, p_tree)
pd.DataFrame([m_log, m_tree])

Decision Tree: exactitud 0.7144 | precisión 0.4103 | recall 0.6648 | F1 0.5074


,modelo,exactitud,precisión,recall,F1
0,Regresión Logística,0.7754,0.4932,0.5658,0.5270
1,Decision Tree,0.7144,0.4103,0.6648,0.5074


**Conclusión:** para detectar la mayor cantidad de clientes que incumplen conviene el **Decision Tree** (recall 0.66 vs 0.57). La regresión logística tiene mejor accuracy (0.78 vs 0.71) y precisión (0.49 vs 0.41).

`class_weight="balanced"` es el parámetro que penaliza más los errores en la clase minoritaria.